## Setup
Run notebooks 01–07 first. Install dependencies once from the project root with `python -m pip install -r requirements.txt`.

In [1]:
from pathlib import Path
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate/"data").is_dir() and (candidate/"notebooks").is_dir(): PROJECT_ROOT=candidate; break
else: raise RuntimeError("Start Jupyter inside the project so its root can be found.")
DATA_DIR=PROJECT_ROOT/"data"; INTERMEDIATE_DIR=DATA_DIR/"02_intermediate"; MODEL_DIR=PROJECT_ROOT/"models"; REPORT_DIR=PROJECT_ROOT/"reports"
INTERMEDIATE_DIR.mkdir(parents=True,exist_ok=True); MODEL_DIR.mkdir(parents=True,exist_ok=True); REPORT_DIR.mkdir(parents=True,exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model


# 10 · Decision engine and operating trade-offs

A score becomes an action only after policy is applied. This notebook compares approve, review, and decline outcomes, estimates manual-review workload, and exposes cost assumptions. Values are illustrative placeholders and require business evidence before use.

### Why reconstruct scores here

This notebook applies policy to the same chronological validation rows and saved artifact used earlier. It rebuilds predictions rather than relying on notebook variables left in memory, making the policy step understandable and reproducible. The validation labels are available for analysis only; a live decision would not know them.

In [ ]:
train_features = pd.read_pickle(INTERMEDIATE_DIR / "train_features.pkl")
artifact = joblib.load(MODEL_DIR / "best_experiment_model.joblib")
feature_cols = list(artifact["feature_cols"])

ordered = train_features.sort_values("TransactionDT", kind="stable").reset_index(drop=True)
validation = ordered.iloc[int(len(ordered) * 0.8):].copy()
X_validation = validation[feature_cols].replace([np.inf, -np.inf], np.nan)
y_validation = validation["isFraud"].astype(int).to_numpy()

if artifact.get("imputer") is not None:
    X_for_model = artifact["imputer"].transform(X_validation)
else:
    X_for_model = X_validation
if artifact.get("scaler") is not None:
    X_for_model = artifact["scaler"].transform(X_for_model)

fraud_probability = artifact["model"].predict_proba(X_for_model)[:, 1]
print(f"Validation sample: {len(y_validation):,}")
print(f"Fraud prevalence: {y_validation.mean():.3%}")

## 1. Set the example operating policy

Low scores are approved, middle scores are reviewed, and high scores are declined. The sample cost proxy treats review as a fixed cost, missed fraud as a loss, and legitimate declines as customer impact. It does not model every downstream outcome.

### Why two cutoffs create three actions

Scores below the lower cutoff are approved, scores between cutoffs are reviewed, and scores above the upper cutoff are declined. A single threshold cannot represent a limited human review channel. The thresholds and costs below are placeholders; each changes the balance between missed fraud, legitimate friction, and queue size.

In [ ]:
# Example operating assumptions: replace with evidence from the business.
APPROVE_BELOW = 0.05
DECLINE_AT_OR_ABOVE = 0.50
REVIEW_COST = 3.00
MISSED_FRAUD_COST = 150.00
LEGITIMATE_DECLINE_COST = 8.00
MAX_REVIEW_RATE = 0.05

assert 0 <= APPROVE_BELOW < DECLINE_AT_OR_ABOVE <= 1

def apply_policy(scores, approve_below, decline_at):
    """Map a score into an approve, review, or decline action."""
    return np.select(
        [scores < approve_below, scores >= decline_at],
        ["approve", "decline"],
        default="review",
    )

actions = apply_policy(
    fraud_probability,
    APPROVE_BELOW,
    DECLINE_AT_OR_ABOVE,
)
pd.Series(actions).value_counts(normalize=True).rename("share").to_frame()

## 2. Summarize decisions, outcomes, and queue size

### Why summarize each action separately

Counts show workload, fraud cases show how outcomes are distributed, and the observed rate helps compare groups. Always read the rate with its denominator: a high rate in a tiny decline group may represent few cases, while a modest rate in a huge approve group can contain most fraud losses.

In [ ]:
outcomes = pd.DataFrame({
    "fraud": y_validation,
    "probability": fraud_probability,
    "action": actions,
})

summary = outcomes.groupby("action").agg(
    transactions=("fraud", "size"),
    fraud_cases=("fraud", "sum"),
    observed_fraud_rate=("fraud", "mean"),
    mean_score=("probability", "mean"),
)
summary["share"] = summary["transactions"] / len(outcomes)
summary

### What this sample cost assumes

The cost proxy charges every review a fixed amount, each fraud approved the average fraud loss, and each legitimate decline a fixed customer-impact cost. It assumes reviewed cases are resolved without residual fraud loss or wrongful declines; that is optimistic unless review outcomes are measured. It also ignores recovery and longer-term customer behavior. Replace the assumptions before using the result.

In [ ]:
reviewed = int((actions == "review").sum())
missed_fraud = int(((actions == "approve") & (y_validation == 1)).sum())
legitimate_declines = int(((actions == "decline") & (y_validation == 0)).sum())

estimated_cost = (
    reviewed * REVIEW_COST
    + missed_fraud * MISSED_FRAUD_COST
    + legitimate_declines * LEGITIMATE_DECLINE_COST
)
review_rate = reviewed / len(y_validation)

print(f"Review queue: {reviewed:,} ({review_rate:.2%})")
print(f"Example capacity ceiling: {MAX_REVIEW_RATE:.2%}")
print(f"Fraud approved: {missed_fraud:,}")
print(f"Legitimate transactions declined: {legitimate_declines:,}")
print(f"Illustrative sample cost: ${estimated_cost:,.2f}")
if review_rate > MAX_REVIEW_RATE:
    print("This policy exceeds the example review capacity.")

## 3. Threshold sensitivity under the example assumptions

This grid compares policies and their review rate. Do not select and report a policy on the same validation data as if it were an independent evaluation.

### Why sweep policy pairs

Searching both cutoffs reveals how queue rate and estimated cost change together. Filtering by review capacity makes the operational constraint explicit. The lowest-cost feasible row is still only a candidate because this window was also used for model selection and labels may be delayed.

In [ ]:
policy_rows = []

for approve_below in np.arange(0.01, 0.31, 0.02):
    first_decline = max(approve_below + 0.05, 0.20)
    for decline_at in np.arange(first_decline, 0.81, 0.05):
        candidate_actions = apply_policy(
            fraud_probability,
            approve_below,
            decline_at,
        )
        review_count = int((candidate_actions == "review").sum())
        fraud_approved = int(
            ((candidate_actions == "approve") & (y_validation == 1)).sum()
        )
        legitimate_declined = int(
            ((candidate_actions == "decline") & (y_validation == 0)).sum()
        )
        cost = (
            fraud_approved * MISSED_FRAUD_COST
            + legitimate_declined * LEGITIMATE_DECLINE_COST
            + review_count * REVIEW_COST
        )
        policy_rows.append({
            "approve_below": approve_below,
            "decline_at": decline_at,
            "review_rate": review_count / len(y_validation),
            "estimated_cost": cost,
        })

policy_grid = pd.DataFrame(policy_rows)
feasible = policy_grid[policy_grid["review_rate"] <= MAX_REVIEW_RATE]
ranked_policies = feasible if not feasible.empty else policy_grid
ranked_policies.sort_values("estimated_cost").head(10)

## 4. Operational controls

A real decision service needs analyst guidance, safe fallbacks, customer appeal and override paths, audit logs, model-version tracking, queue prioritization, and monitoring of action rates. Replace every placeholder with approved estimates and evaluate on a later untouched time period.